# IonQ Debiasing — Noise-Mitigation Comparison

Compare the QPUF (single-stage QPE, `N_TARG=1`, `N_PREC=6`) phase-readout distribution across **three** runs:

1. **Local simulator (noiseless)** — exact statevector of the *same* circuit, the ground-truth reference.
2. **Hardware, no mitigation** — IonQ Forte-Enterprise-1, raw.
3. **Hardware, IonQ debiasing** — IonQ Forte-Enterprise-1 with native `Debias()` error mitigation.

All three use the **identical** Haar-random unitary `U` and seeds stored in the job-result JSON, so they are directly comparable. We score each hardware run against the local simulator with **P(correct bitstring)**, **Total Variation Distance (TVD)**, and **Hellinger fidelity**, and finish with a single plot overlaying all three distributions.

> **Fair-comparison caveat:** IonQ debiasing internally splits the requested shots across symmetrization variants, so its effective per-circuit statistics differ slightly from a plain run at the same shot count. Submit the no-mitig job at the **same `N_SHOTS`** as the debias job for the closest comparison.

In [ ]:
import os
import glob
import json
from collections import defaultdict
from datetime import datetime as _dt

import numpy as np
import matplotlib.pyplot as plt
from qiskit.quantum_info import Statevector

import ionq_noise_mitig as submit   # same-directory submit/calculator module

RESULTS_DIR = "job_results"
PLOTS_DIR = "plots"
os.makedirs(PLOTS_DIR, exist_ok=True)

# --- Which circuit family to compare ------------------------------------------
# Results in job_results/ are grouped by (n_prec, n_targ). The debias vs no-mitig
# split alone does NOT distinguish an N_PREC=6 run from an N_PREC=10 run, so pick
# the family explicitly here, or leave SELECT_N_PREC = None to auto-select the
# family of your most recently submitted job.
SELECT_N_PREC = None   # e.g. 6 or 10 ; None = auto (latest submission's family)
SELECT_N_TARG = None   # e.g. 1 or 2 ; None = infer from SELECT_N_PREC group

# --- Optional manual file overrides (bypass auto-selection entirely) ----------
MITIG_FILE = None      # e.g. "b198eba2-...json"  (hardware, debias ON)
NOMITIG_FILE = None    # e.g. "abcd1234-...json"  (hardware, no mitigation)

## 1. Load, group by circuit family, and classify

Auto-discover every `*.json` in `job_results/`, tag each as **debias** vs **no-mitig**, and group by **`(n_prec, n_targ)`** — because results from different `N_PREC` runs live side by side in the same folder and are otherwise indistinguishable by `circuit_type`. We then select one family (per `SELECT_N_PREC`, or the most recently submitted one) and, within it, the most recent file of each mitigation type. A consistency check refuses to compare a debias/no-mitig pair that aren't the same circuit.

In [ ]:
def _is_debias(res):
    """Classify a job result as debias (True) vs no-mitig (False).

    Priority: explicit `error_mitigation` field -> `circuit_type` suffix ->
    a *non-empty* `sharpenedProbabilities` dict. The last i
    s only a fallback:
    a no-mitig result can still carry a `sharpenedProbabilities` KEY whose value
    is null, so mere key-presence is NOT a reliable signal.
    """
    em = res.get("error_mitigation")
    if em is not None:
        return em == "debias"
    ct = res.get("circuit_type", "")
    if "debias" in ct:
        return True
    if "none" in ct:
        return False
    ionq = (res.get("additional_metadata") or {}).get("ionqMetadata") or {}
    return bool(ionq.get("sharpenedProbabilities"))


def _sortkey(res):
    """Recency key: submission datetime if parseable, else filename."""
    try:
        return (_dt.fromisoformat(res["datetime"]), res["_file"])
    except Exception:
        return (_dt.min, res["_file"])


# --- discover every result and tag it -----------------------------------------
all_results = []
for path in sorted(glob.glob(os.path.join(RESULTS_DIR, "*.json"))):
    res = json.load(open(path))
    res["_file"] = os.path.basename(path)
    res["_mitig"] = "debias" if _is_debias(res) else "nomitig"
    all_results.append(res)
if not all_results:
    raise FileNotFoundError(f"No result JSONs in {RESULTS_DIR}/")

# --- group by (n_prec, n_targ) and report availability ------------------------
families = defaultdict(list)
for r in all_results:
    families[(r["n_prec"], r["n_targ"])].append(r)

print("Available circuit families in job_results/:")
for key in sorted(families):
    grp = families[key]
    nd = sum(r["_mitig"] == "debias" for r in grp)
    nm = sum(r["_mitig"] == "nomitig" for r in grp)
    print(f"  n_prec={key[0]:>2}, n_targ={key[1]}:  debias={nd}  nomitig={nm}")

# --- choose the debias + no-mitig pair ----------------------------------------
by_file = {r["_file"]: r for r in all_results}
if MITIG_FILE or NOMITIG_FILE:
    # manual override: take exactly the named files
    debias_res = by_file.get(MITIG_FILE) if MITIG_FILE else None
    nomitig_res = by_file.get(NOMITIG_FILE) if NOMITIG_FILE else None
else:
    if SELECT_N_PREC is not None:
        cands = [k for k in families if k[0] == SELECT_N_PREC
                 and (SELECT_N_TARG is None or k[1] == SELECT_N_TARG)]
        if not cands:
            raise ValueError(f"No results with n_prec={SELECT_N_PREC}"
                             + (f", n_targ={SELECT_N_TARG}" if SELECT_N_TARG is not None else "")
                             + f". Available: {sorted(families)}")
        fam_key = max(cands, key=lambda k: k[1])     # if >1 n_targ, take the larger
    else:
        latest = max(all_results, key=_sortkey)       # family of newest submission
        fam_key = (latest["n_prec"], latest["n_targ"])
    grp = families[fam_key]
    deb = [r for r in grp if r["_mitig"] == "debias"]
    nom = [r for r in grp if r["_mitig"] == "nomitig"]
    debias_res = max(deb, key=_sortkey) if deb else None
    nomitig_res = max(nom, key=_sortkey) if nom else None

ref_res = debias_res or nomitig_res            # reference for the local sim
if ref_res is None:
    raise FileNotFoundError("No matching result for the selected family.")

# --- never compare mismatched circuits ----------------------------------------
if debias_res and nomitig_res:
    for f in ("n_prec", "n_targ", "seed", "target_init_seed"):
        if debias_res.get(f) != nomitig_res.get(f):
            raise ValueError(f"debias and no-mitig differ in {f!r}: "
                             f"{debias_res.get(f)} vs {nomitig_res.get(f)} -- "
                             "not the same circuit, refusing to compare. "
                             "Set SELECT_N_PREC / MITIG_FILE / NOMITIG_FILE explicitly.")

print(f"\nSelected family: n_prec={ref_res['n_prec']}, n_targ={ref_res['n_targ']}  "
      f"(n_shots={ref_res['n_shots']})")
print(f"  debias  : {debias_res['_file'] if debias_res else '<none>'}")
print(f"  nomitig : {nomitig_res['_file'] if nomitig_res else '<none>'}")
if debias_res is None:
    print("  (no debias result for this family yet -- showing local-sim vs no-mitig)")
if nomitig_res is None:
    print("  (no no-mitig result for this family yet -- showing local-sim vs debias)")

## 2. Local simulator (noiseless)

Rebuild the exact submitted circuit from the stored `unitary` + seeds and run it on a **local statevector simulator**. This noiseless distribution is the reference every hardware run is scored against. The **correct phase bitstring** `b` is its argmax.

Bit-order note: the precision register maps to the measured bitstring as `c[n_prec-1]…c[0]` (qiskit convention) — validated by checking that the local-sim peak coincides with the debiased hardware peak.

In [ ]:
def get_unitary(res):
    u = res["unitary"]
    return np.array(u["real"]) + 1j * np.array(u["imag"])


def local_sim_distribution(res):
    """Exact noiseless precision-register distribution of the submitted circuit."""
    U = get_unitary(res)
    n_prec, n_targ = res["n_prec"], res["n_targ"]
    qc = submit.build_full_circuit(n_prec, n_targ, U, target_init_seed=res["target_init_seed"])
    qc = qc.remove_final_measurements(inplace=False)
    sv = Statevector(qc)
    n = n_prec + n_targ
    # probabilities_dict keys are q_{n-1}...q_0; precision bitstring (display order
    # c[n_prec-1]..c[0]) takes the precision qubits, marginalizing the targets.
    pos = [(n - 1) - (n_targ + c) for c in range(n_prec - 1, -1, -1)]
    marg = defaultdict(float)
    for key, p in sv.probabilities_dict().items():
        marg["".join(key[i] for i in pos)] += p
    return dict(marg)


def counts_to_probs(counts):
    total = sum(counts.values())
    return {k: v / total for k, v in counts.items()}


# Reference circuit comes from whichever result is available (debias or no-mitig);
# both store the same n_prec/n_targ/seed/unitary for the selected family.
local_sim = local_sim_distribution(ref_res)
correct_bits = max(local_sim, key=local_sim.get)
n_prec = ref_res["n_prec"]

print(f"Local simulator (noiseless): {len(local_sim)} bitstrings with nonzero prob")
print(f"Correct phase bitstring b = '{correct_bits}'   local-sim P(b) = {local_sim[correct_bits]:.4f}")

# distributions keyed by label (include only what's available)
dists = {"Local sim": local_sim}
if nomitig_res is not None:
    dists["HW no-mitig"] = counts_to_probs(nomitig_res["counts"])
if debias_res is not None:
    dists["HW debias"] = counts_to_probs(debias_res["counts"])
# consistent series order for all plots/tables
ORDER = [k for k in ["Local sim", "HW no-mitig", "HW debias"] if k in dists]

## 3. Metrics — each hardware run vs the local simulator

- **P(b)** — probability mass on the correct bitstring (higher is better; local-sim value is the ceiling).
- **TVD** = ½·Σ|p−q| — total variation distance to local sim (lower is better).
- **Hellinger fidelity** = (Σ√(pq))² — distribution overlap, 1.0 = identical (higher is better).

In [ ]:
def _align(*ds):
    keys = sorted(set().union(*[d.keys() for d in ds]))
    return [np.array([d.get(k, 0.0) for k in keys]) for d in ds]


def tvd(p, q):
    return float(0.5 * np.sum(np.abs(p - q)))


def hellinger_fidelity(p, q):
    return float(np.sum(np.sqrt(p * q)) ** 2)


metrics = {}
for label in ORDER:
    if label == "Local sim":
        continue
    p_sim, p_hw = _align(local_sim, dists[label])
    metrics[label] = {
        "P(b)": dists[label].get(correct_bits, 0.0),
        "TVD": tvd(p_sim, p_hw),
        "Hellinger fidelity": hellinger_fidelity(p_sim, p_hw),
    }

# pretty table
hdr = f"{'run':14s} {'P(b)':>8} {'TVD':>8} {'Hellinger fid':>14}"
print(hdr); print("-" * len(hdr))
print(f"{'Local sim':14s} {local_sim[correct_bits]:8.4f} {0.0:8.4f} {1.0:14.4f}")
for label in ["HW no-mitig", "HW debias"]:
    if label in metrics:
        m = metrics[label]
        print(f"{label:14s} {m['P(b)']:8.4f} {m['TVD']:8.4f} {m['Hellinger fidelity']:14.4f}")

# verdict
if "HW no-mitig" in metrics and "HW debias" in metrics:
    dt = metrics["HW no-mitig"]["TVD"] - metrics["HW debias"]["TVD"]
    df = metrics["HW debias"]["Hellinger fidelity"] - metrics["HW no-mitig"]["Hellinger fidelity"]
    dp = metrics["HW debias"]["P(b)"] - metrics["HW no-mitig"]["P(b)"]
    print(f"\nDebias vs no-mitig:  TVD {dt:+.4f} ({'better' if dt>0 else 'worse'}),  "
          f"Hellinger fid {df:+.4f},  P(b) {dp:+.4f}")
else:
    print("\n(Submit the no-mitig job to enable the full debias-vs-no-mitig verdict.)")

## 4. Metrics comparison

In [ ]:
hw_labels = [l for l in ["HW no-mitig", "HW debias"] if l in metrics]
colors = {"HW no-mitig": "#d62728", "HW debias": "#2ca02c"}

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
panels = [
    ("P(b)  (higher better)", "P(b)", local_sim[correct_bits]),
    ("TVD to local sim  (lower better)", "TVD", 0.0),
    ("Hellinger fidelity  (higher better)", "Hellinger fidelity", 1.0),
]
for ax, (title, key, ref) in zip(axes, panels):
    vals = [metrics[l][key] for l in hw_labels]
    bars = ax.bar(hw_labels, vals, color=[colors[l] for l in hw_labels])
    ax.axhline(ref, ls="--", color="#1f77b4", lw=1.5, label="local sim")
    ax.set_title(title, fontsize=10)
    ax.tick_params(axis="x", rotation=10)
    ax.legend(fontsize=8)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width() / 2, v, f"{v:.3f}", ha="center", va="bottom", fontsize=9)
fig.suptitle(f"IonQ debias vs no-mitig — QPUF phase readout (b='{correct_bits}')", y=1.03)
fig.tight_layout()
fig.savefig(os.path.join(PLOTS_DIR, "ionq_debias_metrics.png"), dpi=150, bbox_inches="tight")
print("saved:", os.path.join(PLOTS_DIR, "ionq_debias_metrics.png"))
plt.show()

## 5. Appendix — IonQ `sharpenedProbabilities`

Debiasing returns two aggregates: the **averaged** counts (used above as *HW debias*) and a **sharpened** distribution that more aggressively concentrates onto the dominant outcomes. IonQ reports the latter in `ionqMetadata.sharpenedProbabilities`. Note it is indexed over **all qubits** (so keys can be one bit wider than the measured `counts`); shown here as supplementary information, not part of the main comparison.

In [ ]:
sharp = None
if debias_res is not None:
    sharp = (debias_res.get("additional_metadata") or {}).get("ionqMetadata", {}).get("sharpenedProbabilities")
if sharp:
    top = sorted(sharp.items(), key=lambda kv: kv[1], reverse=True)[:8]
    print("IonQ sharpenedProbabilities (top 8):")
    for k, v in top:
        print(f"  {k}  {v:.4f}")
elif debias_res is None:
    print("No debias result selected -- sharpenedProbabilities is debias-only.")
else:
    print("No sharpenedProbabilities in the debias result.")

## 6. All three together — distribution overlay

The headline comparison: **local simulator (noiseless)**, **hardware no-mitig**, and **hardware debias** on one axis, over the bitstrings that carry the most weight in the local simulator. Debiasing should pull the hardware bars back toward the local-sim reference (taller at `b`, lower on the spurious bitstrings).

In [ ]:
TOP_K = 12
# bitstrings to show: the top-K by local-sim weight (the physically meaningful ones)
top_keys = [k for k, _ in sorted(local_sim.items(), key=lambda kv: kv[1], reverse=True)[:TOP_K]]

series_colors = {"Local sim": "#1f77b4", "HW no-mitig": "#d62728", "HW debias": "#2ca02c"}
x = np.arange(len(top_keys))
width = 0.8 / len(ORDER)

fig, ax = plt.subplots(figsize=(max(10, 1.0 * len(top_keys)), 5))
for i, label in enumerate(ORDER):
    vals = [dists[label].get(k, 0.0) for k in top_keys]
    ax.bar(x + (i - (len(ORDER) - 1) / 2) * width, vals, width,
           label=label, color=series_colors[label])

# highlight the correct bitstring
if correct_bits in top_keys:
    ax.axvspan(top_keys.index(correct_bits) - 0.45, top_keys.index(correct_bits) + 0.45,
               color="gold", alpha=0.15, zorder=0)

ax.set_xticks(x)
ax.set_xticklabels(top_keys, rotation=60, ha="right", family="monospace", fontsize=9)
ax.set_xlabel(f"precision bitstring  (correct b='{correct_bits}', highlighted)")
ax.set_ylabel("probability")
ax.set_title(f"QPUF phase readout: local sim vs IonQ hardware (top {TOP_K} bitstrings)")
ax.legend()
fig.tight_layout()
out = os.path.join(PLOTS_DIR, "ionq_debias_3way_comparison.png")
fig.savefig(out, dpi=150, bbox_inches="tight")
print("saved:", out)
if nomitig_res is None:
    print("NOTE: 'HW no-mitig' is missing -- only 2 series shown. Submit that job and re-run "
          "for the full 3-way comparison.")
plt.show()